In [15]:
import pandas as pd
import networkx as nx
from sklearn.model_selection import StratifiedGroupKFold

df = pd.read_csv("../preprocessing/train_data_with_preprocessing2.csv")
df['ctx_node'] = "CTX_" + df['prev_clean'].astype(str)
df['rep_node'] = "REP_" + df['target_clean'].astype(str)

G = nx.Graph()
G.add_edges_from(zip(df['ctx_node'], df['rep_node']))

# --- remove hub nodes (tune the threshold by looking at the degree distribution) ---
HUB_DEGREE = 50
hubs = {n for n, d in G.degree() if d > HUB_DEGREE}
print(f"Removing {len(hubs)} hub nodes")
G.remove_nodes_from(hubs)

node_to_comp = {}
for cid, nodes in enumerate(nx.connected_components(G)):
    for n in nodes:
        node_to_comp[n] = cid

# Assign each row a group: via its non-hub node, else its own singleton group
next_id = len(set(node_to_comp.values()))
groups = []
for i, (c, r) in enumerate(zip(df['ctx_node'], df['rep_node'])):
    if c in node_to_comp:
        groups.append(node_to_comp[c])
    elif r in node_to_comp:
        groups.append(node_to_comp[r])
    else:
        groups.append(next_id + i)       # both nodes are hubs
df['component_id'] = groups

print(df.groupby('component_id').size().sort_values(ascending=False).head(5) / len(df))

# --- stratified + grouped split (~80/20 using 5 folds) ---
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(sgkf.split(df, y=df['label_id'], groups=df['component_id']))

drop_cols = ['ctx_node','rep_node','component_id','split','group','is_toxic','label_id','previous_reply','target_reply']
train_df = df.iloc[train_idx].drop(columns=drop_cols)
val_df   = df.iloc[val_idx].drop(columns=drop_cols)

# sanity check
print(df.iloc[train_idx]['label_id'].value_counts(normalize=True))
print(df.iloc[val_idx]['label_id'].value_counts(normalize=True))
print(len(train_df), len(val_df))

Removing 26 hub nodes
component_id
9     0.150739
10    0.092584
2     0.051413
25    0.041120
5     0.037054
dtype: float64
label_id
0     0.510794
8     0.049608
5     0.049099
6     0.048908
9     0.048335
1     0.047762
10    0.045978
3     0.045978
7     0.045787
4     0.040757
2     0.035280
11    0.031714
Name: proportion, dtype: float64
label_id
0     0.491685
6     0.309818
10    0.048283
4     0.042918
7     0.033798
11    0.031921
3     0.023069
1     0.009925
5     0.007242
9     0.001073
2     0.000268
Name: proportion, dtype: float64
15703 3728


In [1]:
import pandas as pd
import networkx as nx
from sklearn.model_selection import GroupShuffleSplit

# 1. Load your dataset
df = pd.read_csv("../preprocessing/train_data_with_preprocessing2.csv")

# 2. Build the Bipartite Graph
G = nx.Graph()

# Add prefixes to avoid collisions between identical context and reply strings
df['ctx_node'] = "CTX_" + df['prev_clean'].astype(str)
df['rep_node'] = "REP_" + df['target_clean'].astype(str)

# Create edges based on rows
edges = list(zip(df['ctx_node'], df['rep_node']))
G.add_edges_from(edges)

# 3. Find Connected Components
components = list(nx.connected_components(G))

# Map every node to a unique Component ID
node_to_component = {}
for component_id, nodes in enumerate(components):
    for node in nodes:
        node_to_component[node] = component_id

# 4. Map the Component IDs back to the original dataframe
# (You only need to check the context node, as the reply node is guaranteed to have the same ID)
df['component_id'] = df['ctx_node'].map(node_to_component)

# 5. Split the data ensuring components are kept together
# GroupShuffleSplit ensures all rows with the same component_id stay in the same split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

train_idx, val_idx = next(gss.split(df, groups=df['component_id']))

train_df = df.iloc[train_idx].drop(columns=['ctx_node', 'rep_node', 'component_id', 'split', 'group', 'is_toxic', 'label_id', 'previous_reply', 'target_reply'])
val_df = df.iloc[val_idx].drop(columns=['ctx_node', 'rep_node', 'component_id', 'split', 'group', 'is_toxic', 'label_id', 'previous_reply', 'target_reply'])

print(f"Training rows: {len(train_df)}")
print(f"Validation rows: {len(val_df)}")

Training rows: 17991
Validation rows: 1440


In [2]:
train_df.rename(columns={'prev_clean': 'context', 'target_clean': 'reply'}, inplace=True)
val_df.rename(columns={'prev_clean': 'context', 'target_clean': 'reply'}, inplace=True)

from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
  accuracy_score,
  f1_score,
  classification_report,
  confusion_matrix
)

In [3]:
target_col = 'label'

In [4]:
# Extract text columns for TF-IDF
X_context_train = train_df['context'].fillna("").astype(str)
X_reply_train = train_df['reply'].fillna("").astype(str)
y_train = train_df[target_col].fillna("nan").astype(str)

X_context_dev = val_df['context'].fillna("").astype(str)
X_reply_dev = val_df['reply'].fillna("").astype(str)
y_dev = val_df[target_col].fillna("nan").astype(str)

In [5]:
# Word-level TF-IDF
context_word = TfidfVectorizer(
  analyzer="word",
  ngram_range=(1, 2),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

reply_word = TfidfVectorizer(
  analyzer="word",
  ngram_range=(1, 2),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

In [6]:
# Character-level TF-IDF
context_char = TfidfVectorizer(
  analyzer="char_wb",
  ngram_range=(3, 5),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

reply_char = TfidfVectorizer(
  analyzer="char_wb",
  ngram_range=(3, 5),
  min_df=2,
  max_features=50000,
  sublinear_tf=True
)

In [7]:
# Fit only on training data, transform both
Cw_train = context_word.fit_transform(X_context_train)
Cw_dev = context_word.transform(X_context_dev)

Rw_train = reply_word.fit_transform(X_reply_train)
Rw_dev = reply_word.transform(X_reply_dev)

Cc_train = context_char.fit_transform(X_context_train)
Cc_dev = context_char.transform(X_context_dev)

Rc_train = reply_char.fit_transform(X_reply_train)
Rc_dev = reply_char.transform(X_reply_dev)

In [8]:
# Combine all four sparse feature matrices
X_train = hstack([Cw_train, Cc_train, Rw_train, Rc_train], format="csr")
X_dev = hstack([Cw_dev, Cc_dev, Rw_dev, Rc_dev], format="csr")

print("Training feature matrix:", X_train.shape)
print("Development feature matrix:", X_dev.shape)

Training feature matrix: (17991, 36109)
Development feature matrix: (1440, 36109)


In [10]:
model = MultinomialNB(alpha=0.5, fit_prior=True)

model.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",0.5
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](12,)","[ 97., 553., 781.,..., 852., 537.,9636.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](12,)","[-5.22,-3.48,-3.14,...,-3.05,-3.51,-0.62]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[<U35](12,)","['Dehumanization','Harassment','Hate Speech',...,'Threat', 'Toxic Endorsement / Agreement','nan']"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](12, 36109)","[[ 0. , 0. , 3.03,..., 0. , 0. , 0.09], [ 0. , 0. , 0.19,..., 0. , 0. , 0. ], [ 0. , 0. , 1.85,..., 0.07, 0.07, 0.2 ], ..., [ 0. , 0. , 18.53,..., 0. , 0. , 0.26], [ 0. , 0. , 0.73,..., 0. , 0. , 0. ], [ 4.91, 4.91,140.93,..., 0. , 0. , 4.94]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](12, 36109)","[[-10.63,-10.63, -8.68,...,-10.63,-10.63,-10.46], [-11.11,-11.11,-10.79,...,-11.11,-11.11,-11.11], [-11.3 ,-11.3 , -9.75,...,-11.17,-11.17,-10.97], ..., [-11.29,-11.29, -7.65,...,-11.29,-11.29,-10.88], [-11.11,-11.11,-10.2 ,...,-11.11,-11.11,-11.11], [-10.91,-10.91, -7.65,...,-13.29,-13.29,-10.91]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,36109


In [11]:
accuracy = accuracy_score(y_dev, model.predict(X_dev))
f1 = f1_score(y_dev, model.predict(X_dev), average='weighted')
classification_rep = classification_report(y_dev, model.predict(X_dev))

print("Development Accuracy:", accuracy)
print("Development F1 Score:", f1)
print("Classification Report:\n", classification_rep)

Development Accuracy: 0.44722222222222224
Development F1 Score: 0.4092589363074778
Classification Report:
                                      precision    recall  f1-score   support

                     Dehumanization       0.62      0.04      0.08       690
                         Harassment       0.40      1.00      0.57         2
                        Hate Speech       0.05      0.67      0.10        27
                    Identity Attack       0.94      0.98      0.96        95
 Incitement / Encouragement of Harm       0.01      0.50      0.01         2
                             Insult       0.38      0.92      0.54       130
                 Mockery / Ridicule       0.93      0.88      0.90        81
                          Profanity       0.18      0.08      0.11        25
Sexual Harassment / Objectification       0.14      0.03      0.04        40
                             Threat       0.79      0.90      0.84        50
      Toxic Endorsement / Agreement       0.8

In [17]:
import optuna
import numpy as np
from sklearn.metrics import log_loss

def objective(trial):
  alpha = trial.suggest_float("alpha", 0.1, 1.0)
  fit_prior = trial.suggest_categorical("fit_prior", [True, False])

  model = MultinomialNB(
    alpha=alpha,
    fit_prior=fit_prior,
  )

  model.fit(X_train, y_train)
  y_pred = model.predict(X_dev)
  score = accuracy_score(y_dev, y_pred) 
  # y_proba = model.predict_proba(X_dev)
  # score = log_loss(y_dev, y_proba)  # Use log loss for evaluation

  return score

In [18]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

[I 2026-11-03 00:32:35,421] A new study created in memory with name: no-name-a4489785-0e88-4ec8-a86e-3f8c63bac26f
[I 2026-11-03 00:32:35,526] Trial 0 finished with value: 0.44722222222222224 and parameters: {'alpha': 0.5238166029722439, 'fit_prior': True}. Best is trial 0 with value: 0.44722222222222224.
[I 2026-11-03 00:32:35,604] Trial 1 finished with value: 0.6145833333333334 and parameters: {'alpha': 0.24249630209730594, 'fit_prior': True}. Best is trial 1 with value: 0.6145833333333334.
[I 2026-11-03 00:32:35,684] Trial 2 finished with value: 0.625 and parameters: {'alpha': 0.2895897482203523, 'fit_prior': False}. Best is trial 2 with value: 0.625.
[I 2026-11-03 00:32:35,760] Trial 3 finished with value: 0.6020833333333333 and parameters: {'alpha': 0.2523978984831138, 'fit_prior': True}. Best is trial 2 with value: 0.625.
[I 2026-11-03 00:32:35,836] Trial 4 finished with value: 0.44513888888888886 and parameters: {'alpha': 0.6552142736546858, 'fit_prior': False}. Best is trial 2 w

In [19]:
study.best_params

{'alpha': 0.10014685426190002, 'fit_prior': False}

In [20]:
study.best_value

0.8013888888888889

In [9]:
best_model = MultinomialNB(
  alpha = 0.1,
  fit_prior = False
)

best_model.fit(X_train, y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",0.1
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",False
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](12,)","[ 97., 553., 781.,..., 852., 537.,9636.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](12,)","[-2.48,-2.48,-2.48,...,-2.48,-2.48,-2.48]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[<U35](12,)","['Dehumanization','Harassment','Hate Speech',...,'Threat', 'Toxic Endorsement / Agreement','nan']"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](12, 36109)","[[ 0. , 0. , 3.03,..., 0. , 0. , 0.09], [ 0. , 0. , 0.19,..., 0. , 0. , 0. ], [ 0. , 0. , 1.85,..., 0.07, 0.07, 0.2 ], ..., [ 0. , 0. , 18.53,..., 0. , 0. , 0.26], [ 0. , 0. , 0.73,..., 0. , 0. , 0. ], [ 4.91, 4.91,140.93,..., 0. , 0. , 4.94]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](12, 36109)","[[-11.05,-11.05, -7.61,...,-11.05,-11.05,-10.39], [-12.16,-12.16,-11.1 ,...,-12.16,-12.16,-12.16], [-12.47,-12.47, -9.5 ,...,-11.95,-11.95,-11.38], ..., [-12.45,-12.45, -7.22,...,-12.45,-12.45,-11.18], [-12.15,-12.15,-10.03,...,-12.15,-12.15,-12.15], [-10.94,-10.94, -7.6 ,...,-14.85,-14.85,-10.93]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,36109


In [10]:
print("Best Model Development Accuracy:", accuracy_score(y_dev, best_model.predict(X_dev)))
print("Best Model Development F1 Score:", f1_score(y_dev, best_model.predict(X_dev), average='weighted'))
print("Best Model Classification Report:\n", classification_report(y_dev, best_model.predict(X_dev)))

Best Model Development Accuracy: 0.8013888888888889
Best Model Development F1 Score: 0.7936609354361328
Best Model Classification Report:
                                      precision    recall  f1-score   support

                     Dehumanization       0.78      0.90      0.83       690
                         Harassment       0.50      1.00      0.67         2
                        Hate Speech       0.21      0.48      0.29        27
                    Identity Attack       0.93      0.98      0.95        95
 Incitement / Encouragement of Harm       0.00      0.00      0.00         2
                             Insult       0.78      0.50      0.61       130
                 Mockery / Ridicule       1.00      0.86      0.93        81
                          Profanity       0.00      0.00      0.00        25
Sexual Harassment / Objectification       0.25      0.03      0.05        40
                             Threat       0.98      0.90      0.94        50
      Toxic E

In [11]:
df_test = pd.read_csv("../preprocessing/test_data_with_preprocessing2.csv")
y_test = df_test[target_col].fillna("nan").astype(str)
X_test_context = df_test['context_clean'].fillna("").astype(str)
X_test_reply = df_test['reply_clean'].fillna("").astype(str)

In [12]:
Cw_test = context_word.transform(X_test_context)
Rw_test = reply_word.transform(X_test_reply)

Cc_test = context_char.transform(X_test_context)
Rc_test = reply_char.transform(X_test_reply)

X_test = hstack([Cw_test, Cc_test, Rw_test, Rc_test], format="csr")

In [13]:
print("Test Accuracy:", accuracy_score(y_test, best_model.predict(X_test)))

Test Accuracy: 0.06208425720620843


In [14]:
f1_score(y_test, best_model.predict(X_test), average='weighted')

0.08917124152258665